# Hallucination Probe - Part 2: Improvements

Builds on notebook 01. What is new:

| # | Improvement | Question it answers |
|---|---|---|
| 1 | **Google Drive storage**, resumable everywhere | Nothing is lost on disconnect |
| 2 | **5,000 questions** + stricter grading (word-boundary match) | Is the result significant with more data? |
| 3 | **Three poolings**: answer-mean, answer-last-token, **prompt-last-token** | Can we predict a hallucination *before* the model answers? |
| 4 | **Layer chosen on validation data**, never on test | Is our AUROC unbiased? |
| 5 | **Baselines**: log-prob, and probe + log-prob combined, with significance tests | Does the probe really beat the free option? |
| 6 | **Self-consistency** on a subset, with **measured time per question** | How much cheaper is our method? |
| 7 | **Cross-dataset transfer**: train on TriviaQA, test on Natural Questions | Did the probe learn "guessing" or just TriviaQA? |
| 8 | **Multi-model**: change `MODEL_ID`, re-run, compare | Does the signal grow with model size? Same relative depth? |
| 9 | **Gradio demo** with a public link | Live demo for the review |

**Runtime -> Change runtime type -> T4 GPU.** Run cells in order.
Approximate time on a T4 for the 1.5B model: generation ~15 min, extraction ~12 min,
probes ~10 min (CPU), self-consistency ~8 min.

## Stage 0 - Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn matplotlib gradio

import json, re, string, time, random
from collections import Counter
from pathlib import Path
import numpy as np, torch
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/hallucination_probe_v2")
except Exception as e:
    ROOT = Path("/content/hallucination_probe_v2")
    print("!!! WARNING: Drive NOT mounted - results will be LOST on disconnect !!!", e)

In [ ]:
# ---- change these, nothing else ------------------------------------------
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # next: "Qwen/Qwen2.5-3B-Instruct", "meta-llama/Llama-3.2-1B-Instruct",
                                          #       "meta-llama/Llama-3.2-3B-Instruct", "google/gemma-2-2b-it"
# For cross-model transfer (notebook 04) every model MUST use the same N_TRIVIA, so questions pair up.
N_TRIVIA = 5000          # TriviaQA questions (train + test); 3000 if short on time
N_NQ = 1000              # Natural Questions (transfer test only)
MAX_NEW_TOKENS = 20
BATCH = 8
SHARD_SIZE = 250
N_SEEDS = 10             # repeated splits for error bars
SC_N, SC_K = 300, 10     # self-consistency: questions, samples per question
POOLINGS = ["a_mean", "a_last", "p_last"]
POOL_NAMES = {"a_mean": "mean over answer tokens (notebook 01)",
              "a_last": "last answer token",
              "p_last": "last prompt token - BEFORE the answer exists"}
# --------------------------------------------------------------------------

WORK = ROOT / MODEL_ID.split("/")[-1]      # one folder per model
WORK.mkdir(parents=True, exist_ok=True)
print("saving to", WORK)

RESULTS_FILE = WORK / "results.json"
results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
results["model"] = MODEL_ID
def save_results():
    RESULTS_FILE.write_text(json.dumps(results, indent=2))

TIMINGS_FILE = WORK / "timings.json"
timings = json.loads(TIMINGS_FILE.read_text()) if TIMINGS_FILE.exists() else {}
def save_timings():
    TIMINGS_FILE.write_text(json.dumps(timings, indent=2))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16))
model.eval()
N_LAYERS, HIDDEN = model.config.num_hidden_layers, model.config.hidden_size
results["n_layers"], results["hidden"] = N_LAYERS, HIDDEN
print(N_LAYERS, "layers, hidden dim", HIDDEN)

### Grading (v3)

Two fixes over notebook 01:
- an alias that already appears in the question does not count (the v2 fix);
- **word-boundary** match instead of plain substring, so `"us"` no longer matches `"bus"`,
  which also lets short answers like `"8"` or `"UK"` be graded instead of always marked wrong.

In [ ]:
PUNCT = set(string.punctuation)

def normalize(s):
    s = s.lower()
    s = "".join(c for c in s if c not in PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def is_correct(pred, aliases, question):
    p, q = normalize(pred), normalize(question)
    if not p:
        return False
    for a in aliases:
        na = normalize(a)
        if not na or re.search(r"\b" + re.escape(na) + r"\b", q):
            continue                              # echoed from the question
        if re.search(r"\b" + re.escape(na) + r"\b", p):
            return True
    return False

def build_prompt(question):
    msgs = [{"role": "user",
             "content": "Answer with just the answer, no explanation.\nQ: " + question + "\nA:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

assert is_correct("Paris, France", ["Paris"], "Capital of France?")
assert not is_correct("a bus", ["US"], "Which country?")
assert is_correct("8", ["8", "eight"], "How many legs does a spider have?")
print("grading tests passed")

## Stage 1 - Ask the questions and grade the answers

In [ ]:
from datasets import load_dataset

def load_trivia(n):
    ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
    ds = ds.select(range(min(n, len(ds))))
    return [{"question": r["question"],
             "aliases": r["answer"]["aliases"] or [r["answer"]["value"]]} for r in ds]

def load_nq(n):
    try:
        ds = load_dataset("google-research-datasets/nq_open", split="validation")
    except Exception:
        ds = load_dataset("nq_open", split="validation")
    ds = ds.select(range(min(n, len(ds))))
    return [{"question": r["question"] + "?", "aliases": list(r["answer"])} for r in ds]

trivia_items, nq_items = load_trivia(N_TRIVIA), load_nq(N_NQ)
print(len(trivia_items), "TriviaQA |", len(nq_items), "Natural Questions")
print(trivia_items[0]); print(nq_items[0])

In [ ]:
def generate_answers(name, items):
    path = WORK / f"{name}_answers.jsonl"
    done = sum(1 for _ in open(path, encoding="utf-8")) if path.exists() else 0
    print(name, "- resuming from", done)
    t0, n_new = time.time(), 0
    with open(path, "a", encoding="utf-8") as out:
        for start in tqdm(range(done, len(items), BATCH), desc=name):
            rows = items[start:start + BATCH]
            prompts = [build_prompt(r["question"]) for r in rows]
            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            answers = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
            for r, p, a in zip(rows, prompts, answers):
                a = a.strip()
                out.write(json.dumps({
                    "question": r["question"], "prompt": p, "answer": a, "aliases": r["aliases"],
                    "label": 0 if is_correct(a, r["aliases"], r["question"]) else 1,
                    "group": normalize(r["aliases"][0])}, ensure_ascii=False) + "\n")
                n_new += 1
            out.flush()
    if n_new:
        timings["generate_s_per_q"] = (time.time() - t0) / n_new
        save_timings()
    recs = [json.loads(l) for l in open(path, encoding="utf-8")]
    rate = float(np.mean([r["label"] for r in recs]))
    print(f"{name}: {len(recs)} graded - hallucination rate {rate:.3f} (accuracy {1 - rate:.3f})")
    results[f"{name}_accuracy"] = 1 - rate; save_results()
    return recs

trivia_recs = generate_answers("trivia", trivia_items)
nq_recs = generate_answers("nq", nq_items)

### Audit - count your disagreements and report the rate

In [ ]:
random.seed(1)
for i, r in enumerate(random.sample(trivia_recs, 20), 1):
    print(f"[{i:2d}] graded {'WRONG' if r['label'] else 'right'}")
    print("     Q:", r["question"])
    print("     model:", repr(r["answer"]))
    print("     gold :", r["aliases"][:5], "\n")

## Stage 2 - Extract activations (3 poolings + log-probs in ONE forward pass)

Per question and per layer we save three vectors:
- `a_mean` - mean over answer tokens (what notebook 01 used)
- `a_last` - the last answer token
- `p_last` - the last prompt token. **The model has not produced the answer yet** here,
  so if this works we can flag a likely hallucination *before generating it*.

The token log-probabilities for the baseline come from the same forward pass, so the
baseline costs no extra compute. Prompt and answer are tokenised separately and
concatenated, so the prompt/answer boundary is exact.

In [ ]:
@torch.no_grad()
def features_for(prompt, answer):
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    a_ids = tok(answer, add_special_tokens=False).input_ids
    if not a_ids:
        return None
    ids = torch.tensor([p_ids + a_ids], device=model.device)
    out = model(input_ids=ids, output_hidden_states=True)
    n = len(p_ids)
    hs = torch.stack(out.hidden_states)[:, 0]          # (layers+1, tokens, hidden)
    ans = hs[:, n:]
    feats = {"a_mean": ans.mean(1), "a_last": ans[:, -1], "p_last": hs[:, n - 1]}
    feats = {k: v.float().cpu().numpy().astype(np.float16) for k, v in feats.items()}
    lp = (out.logits[0, n - 1:-1].float().log_softmax(-1)
          .gather(1, ids[0, n:].unsqueeze(1)).squeeze(1))
    return feats, float(lp.mean()), float(lp.min())

def extract(name, recs):
    t0, n_new = time.time(), 0
    for s in range(0, len(recs), SHARD_SIZE):
        path = WORK / f"{name}_feats_{s // SHARD_SIZE:05d}.npz"
        if path.exists():
            continue
        F = {k: [] for k in POOLINGS}
        meta = {"label": [], "group": [], "mean_lp": [], "min_lp": [], "idx": []}
        for i in tqdm(range(s, min(s + SHARD_SIZE, len(recs))), desc=f"{name} shard {s // SHARD_SIZE}"):
            r = recs[i]
            res = features_for(r["prompt"], r["answer"])
            n_new += 1
            if res is None:
                continue
            f, mlp, nlp = res
            for k in POOLINGS:
                F[k].append(f[k])
            for k, v in zip(meta, (r["label"], r["group"], mlp, nlp, i)):
                meta[k].append(v)
        empty = np.zeros((0, N_LAYERS + 1, HIDDEN), np.float16)
        np.savez(path, **{k: np.stack(v) if v else empty for k, v in F.items()},
                 label=np.array(meta["label"], dtype=np.int64),
                 group=np.array(meta["group"], dtype=str),
                 mean_lp=np.array(meta["mean_lp"]), min_lp=np.array(meta["min_lp"]),
                 idx=np.array(meta["idx"], dtype=np.int64))
    if n_new:
        timings["probe_forward_s_per_q"] = (time.time() - t0) / n_new
        save_timings()
    print(name, "extraction done")

def load_feats(name):
    parts = [np.load(p) for p in sorted(WORK.glob(f"{name}_feats_*.npz"))]
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}

extract("trivia", trivia_recs)
extract("nq", nq_recs)
print("GPU only needed again for self-consistency and the demo")

## Stage 3 - Which pooling, which layer?

**Unbiased layer choice:** the training data is split again into train/validation, the layer
is picked on validation, and the test set is only used afterwards. (Notebook 01 picked the
best layer *on the test set*, which slightly flatters the number.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def fit(X, y):
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)).fit(X, y)

def risk_of(clf, X):
    return clf.predict_proba(X)[:, 1]

def split(y, g, seed, test=0.3):
    return next(GroupShuffleSplit(1, test_size=test, random_state=seed)
                .split(np.zeros(len(y)), y, g))

def choose_layer(X, y, g):
    tr, _ = split(y, g, 0)                         # test set untouched
    itr, iva = split(y[tr], g[tr], 1, test=0.25)
    tr_, va = tr[itr], tr[iva]
    val = []
    for L in tqdm(range(X.shape[1]), leave=False):
        XL = X[:, L].astype(np.float32)
        val.append(roc_auc_score(y[va], risk_of(fit(XL[tr_], y[tr_]), XL[va])))
    return int(np.argmax(val)), val

D = load_feats("trivia")
y, g = D["label"], D["group"]
print("TriviaQA:", len(y), "examples, hallucination rate", round(float(y.mean()), 3))

layers, sweeps = {}, {}
for pool in POOLINGS:
    L, val = choose_layer(D[pool], y, g)
    layers[pool], sweeps[pool] = L, val
    print(f"{pool:7s} best layer {L:2d} ({100 * L / N_LAYERS:.0f}% depth)  "
          f"val AUROC {val[L]:.4f}   <- {POOL_NAMES[pool]}")

plt.figure(figsize=(8, 4))
for pool in POOLINGS:
    plt.plot(sweeps[pool], marker="o", ms=3, label=POOL_NAMES[pool])
plt.axhline(0.5, ls="--", lw=1, c="#999", label="chance")
plt.xlabel("layer"); plt.ylabel("validation AUROC"); plt.grid(alpha=.25); plt.legend(fontsize=8)
plt.title(f"Layer sweep by pooling - {MODEL_ID}")
plt.tight_layout(); plt.savefig(WORK / "layer_sweep_poolings.png", dpi=160); plt.show()

results["layers"], results["sweeps"] = layers, sweeps; save_results()

### Error bars: 10 entity-disjoint splits per pooling, plus the leakage check

In [ ]:
rob = {}
for pool in POOLINGS:
    XL = D[pool][:, layers[pool]].astype(np.float32)
    s = []
    for seed in range(N_SEEDS):
        tr, te = split(y, g, seed)
        s.append(roc_auc_score(y[te], risk_of(fit(XL[tr], y[tr]), XL[te])))
    rob[pool] = s
    print(f"{pool:7s} layer {layers[pool]:2d}   test AUROC {np.mean(s):.4f} +/- {np.std(s):.4f}")

BEST = max(POOLINGS, key=lambda k: np.mean(rob[k]))
L = layers[BEST]
Xb = D[BEST][:, L].astype(np.float32)
print(f"\nBEST pooling: {BEST} ({POOL_NAMES[BEST]}), layer {L}")
print("p_last vs a_mean: %+.4f  <- can we predict BEFORE answering?"
      % (np.mean(rob["p_last"]) - np.mean(rob["a_mean"])))

rnd = []
for seed in range(N_SEEDS):
    perm = np.random.default_rng(seed).permutation(len(y)); cut = int(0.7 * len(y))
    rnd.append(roc_auc_score(y[perm[cut:]], risk_of(fit(Xb[perm[:cut]], y[perm[:cut]]), Xb[perm[cut:]])))
d, noise = np.mean(rnd) - np.mean(rob[BEST]), np.hypot(np.std(rnd), np.std(rob[BEST]))
print(f"random split {np.mean(rnd):.4f} vs entity split {np.mean(rob[BEST]):.4f}: "
      f"{d:+.4f} -> {'within noise: NO leakage' if abs(d) < noise else 'real gap - investigate'}")

results["robustness"] = {k: {"mean": float(np.mean(v)), "std": float(np.std(v))} for k, v in rob.items()}
results["best_pooling"], results["best_layer"] = BEST, L
results["best_rel_depth"] = L / N_LAYERS
results["random_split_auroc"] = float(np.mean(rnd)); save_results()

### Negative control: shuffled labels must give chance (~0.50)

If the probe still scores well on randomly permuted labels, something in the pipeline is leaking. Also saves the probe now, so `03_demo.ipynb` works even if the later stages are skipped.

In [ ]:
import joblib
shuf = []
for seed in range(N_SEEDS):
    tr, te = split(y, g, seed)
    y_perm = np.random.default_rng(100 + seed).permutation(y[tr])
    shuf.append(roc_auc_score(y[te], risk_of(fit(Xb[tr], y_perm), Xb[te])))
print(f"shuffled-label AUROC {np.mean(shuf):.4f} +/- {np.std(shuf):.4f}  "
      f"-> {'PASS (chance)' if abs(np.mean(shuf) - 0.5) < 0.05 else 'FAIL - investigate leakage'}")
results["shuffled_label_auroc"] = {"mean": float(np.mean(shuf)), "std": float(np.std(shuf))}
save_results()

joblib.dump({"probe": fit(Xb, y), "pooling": BEST, "layer": L, "model": MODEL_ID,
             "auroc": float(np.mean(rob[BEST]))}, WORK / "probe.joblib")
print("probe saved ->", WORK / "probe.joblib")


## Stage 4 - Does the probe beat the free baseline?

Same 10 test splits for every method, so the comparison is paired and fair.
Also tests **probe + log-prob combined**: if it beats both, they carry different information.

In [ ]:
lpf = np.stack([D["mean_lp"], D["min_lp"]], 1).astype(np.float32)
Xc = np.hstack([Xb, lpf])
rows = {"mean token log-prob": [], "min token log-prob": [],
        "internal-state probe": rob[BEST], "probe + log-prob": []}
for seed in range(N_SEEDS):
    tr, te = split(y, g, seed)
    rows["mean token log-prob"].append(roc_auc_score(y[te], -D["mean_lp"][te]))
    rows["min token log-prob"].append(roc_auc_score(y[te], -D["min_lp"][te]))
    rows["probe + log-prob"].append(roc_auc_score(y[te], risk_of(fit(Xc[tr], y[tr]), Xc[te])))

print("%-24s %18s" % ("method", "AUROC (10 splits)"))
print("-" * 44)
for k, v in rows.items():
    print("%-24s %10.4f +/- %.4f" % (k, np.mean(v), np.std(v)))

LP = max(["mean token log-prob", "min token log-prob"], key=lambda k: np.mean(rows[k]))
diff = np.array(rows["internal-state probe"]) - np.array(rows[LP])
print(f"\nprobe beats '{LP}' on {int((diff > 0).sum())}/{N_SEEDS} splits, mean gain {diff.mean():+.4f}")

# paired bootstrap on split 0
tr, te = split(y, g, 0)
ps = risk_of(fit(Xb[tr], y[tr]), Xb[te])
ls = -D["min_lp"][te] if LP.startswith("min") else -D["mean_lp"][te]
rng, diffs = np.random.default_rng(0), []
for _ in range(2000):
    i = rng.integers(0, len(te), len(te))
    if len(set(y[te][i])) == 2:
        diffs.append(roc_auc_score(y[te][i], ps[i]) - roc_auc_score(y[te][i], ls[i]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"bootstrap (test n={len(te)}): probe - logprob = {np.mean(diffs):+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]"
      f"  -> {'SIGNIFICANT' if lo > 0 else 'not significant'}")

results["baselines"] = {k: {"mean": float(np.mean(v)), "std": float(np.std(v))} for k, v in rows.items()}
results["bootstrap_ci"] = [float(lo), float(hi)]; save_results()

## Stage 5 - Selective answering: probe vs log-prob vs a perfect oracle

In [ ]:
def risk_coverage(risk, wrong):
    order = np.argsort(risk)
    acc = np.cumsum(1 - wrong[order]) / np.arange(1, len(order) + 1)
    return np.arange(1, len(order) + 1) / len(order), acc

wrong = y[te]; base = 1 - wrong.mean()
cov, acc_p = risk_coverage(ps, wrong)
_, acc_l = risk_coverage(ls, wrong)

print("coverage   probe  logprob  oracle  captured")
sel = {}
for c in (0.9, 0.8, 0.7, 0.6, 0.5):
    i = max(0, int(c * len(cov)) - 1); o = min(1.0, base / c)
    cap = (acc_p[i] - base) / (o - base)
    sel[str(c)] = {"probe": float(acc_p[i]), "logprob": float(acc_l[i]), "oracle": o, "captured": float(cap)}
    print(f"  {100*c:3.0f}%   {100*acc_p[i]:5.1f}%  {100*acc_l[i]:5.1f}%  {100*o:5.1f}%   {100*cap:4.0f}%")

plt.figure(figsize=(7, 4))
plt.plot(100 * cov, 100 * acc_p, lw=2, label="internal-state probe")
plt.plot(100 * cov, 100 * acc_l, lw=1.5, label=LP)
plt.plot(100 * cov, 100 * np.minimum(1, base / cov), lw=1, ls=":", c="k", label="perfect oracle")
plt.axhline(100 * base, ls="--", lw=1, c="#a4551b", label=f"answer everything ({100*base:.1f}%)")
plt.xlabel("% of questions answered"); plt.ylabel("accuracy on answered (%)")
plt.legend(); plt.grid(alpha=.25); plt.title("Abstaining on risky questions")
plt.tight_layout(); plt.savefig(WORK / "risk_coverage_v2.png", dpi=160); plt.show()
results["selective"] = sel; save_results()

## Stage 6 - Cross-dataset transfer: TriviaQA -> Natural Questions

The probe is trained **only on TriviaQA** and tested on Natural Questions it has never seen.
If AUROC stays high, the probe learned a general "the model is guessing" signal,
not TriviaQA-specific patterns.

In [ ]:
N = load_feats("nq")
yn, gn = N["label"], N["group"]
Xn = N[BEST][:, L].astype(np.float32)
print("NQ:", len(yn), "examples, hallucination rate", round(float(yn.mean()), 3))

transfer = roc_auc_score(yn, risk_of(fit(Xb, y), Xn))       # train on ALL TriviaQA
indomain = []
for seed in range(N_SEEDS):
    trn, ten = split(yn, gn, seed)
    indomain.append(roc_auc_score(yn[ten], risk_of(fit(Xn[trn], yn[trn]), Xn[ten])))
lp_nq = roc_auc_score(yn, -N["min_lp"])

print(f"\nprobe trained on TriviaQA, tested on NQ : {transfer:.4f}")
print(f"probe trained on NQ itself (10 splits)   : {np.mean(indomain):.4f} +/- {np.std(indomain):.4f}")
print(f"min log-prob on NQ (no training)         : {lp_nq:.4f}")
results["transfer_nq"] = {"trivia_to_nq": float(transfer), "nq_in_domain": float(np.mean(indomain)),
                          "min_logprob": float(lp_nq)}; save_results()

## Stage 7 - Self-consistency: accuracy AND cost

Sample `SC_K` answers per question; if they disagree with the greedy answer, the model is
probably guessing. Run on `SC_N` test questions (the same ones the probe is scored on),
and **time it** so we can report the cost honestly.

In [ ]:
sub = te[:SC_N]
sub_recs = [trivia_recs[D["idx"][i]] for i in sub]
SC_BATCH = 4
agree, major = [], []
t0 = time.time()
for b in tqdm(range(0, len(sub_recs), SC_BATCH), desc="self-consistency"):
    chunk = sub_recs[b:b + SC_BATCH]
    enc = tok([r["prompt"] for r in chunk], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                             temperature=0.7, top_p=0.9, num_return_sequences=SC_K,
                             pad_token_id=tok.pad_token_id)
    dec = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    for j, r in enumerate(chunk):
        samples = [normalize(t.strip()) for t in dec[j * SC_K:(j + 1) * SC_K]]
        agree.append(sum(s == normalize(r["answer"]) for s in samples) / SC_K)
        major.append(Counter(samples).most_common(1)[0][1] / SC_K)
timings["sc_s_per_q"] = (time.time() - t0) / len(sub_recs); save_timings()

ysub = y[sub]
sc_rows = {
    f"self-consistency, agree w/ greedy (K={SC_K})": roc_auc_score(ysub, -np.array(agree)),
    f"self-consistency, majority share (K={SC_K})": roc_auc_score(ysub, -np.array(major)),
    "min token log-prob": roc_auc_score(ysub, -D["min_lp"][sub]),
    "internal-state probe": roc_auc_score(ysub, ps[:SC_N]),
}
gen_s = timings.get("generate_s_per_q", float("nan"))
probe_s = timings.get("probe_forward_s_per_q", float("nan"))
sc_s = timings["sc_s_per_q"]
print(f"\nSame {len(sub)} test questions:")
for k, v in sc_rows.items():
    print(f"  {k:45s} AUROC {v:.4f}")
print(f"\nExtra cost per question on top of answering it ({gen_s:.3f}s):")
print(f"  probe            {probe_s:.3f}s  (one forward pass; log-prob comes free from it)")
print(f"  self-consistency {sc_s:.3f}s  -> {sc_s / probe_s:.0f}x the probe")
results["self_consistency"] = {"auroc": sc_rows, "sc_s_per_q": sc_s,
                               "probe_s_per_q": probe_s, "generate_s_per_q": gen_s}
save_results()

## Stage 8 - Compare models

After running this notebook for one model, change `MODEL_ID` in the config cell
(e.g. `"Qwen/Qwen2.5-3B-Instruct"`) and **Runtime -> Run all** again. Each model gets
its own folder. This cell reads every folder and compares them.

*Llama and Gemma are gated: accept the licence on huggingface.co and log in with
`from huggingface_hub import login; login()` first.*

In [ ]:
runs = [json.loads(p.read_text()) for p in sorted(ROOT.glob("*/results.json"))]
runs = [r for r in runs if "robustness" in r]
print("%-30s %6s %8s %10s %15s %9s %9s" % ("model", "layers", "acc", "pooling",
                                          "AUROC", "layer", "depth"))
for r in runs:
    b = r["best_pooling"]
    print("%-30s %6d %7.1f%% %10s %8.4f+/-%.3f %9d %8.0f%%" % (
        r["model"].split("/")[-1], r["n_layers"], 100 * r.get("trivia_accuracy", float("nan")), b,
        r["robustness"][b]["mean"], r["robustness"][b]["std"], r["best_layer"], 100 * r["best_rel_depth"]))

plt.figure(figsize=(7, 4))
for r in runs:
    s = r["sweeps"][r["best_pooling"]]
    plt.plot(np.arange(len(s)) / (len(s) - 1) * 100, s, marker="o", ms=2, label=r["model"].split("/")[-1])
plt.xlabel("relative depth (%)"); plt.ylabel("validation AUROC"); plt.grid(alpha=.25); plt.legend()
plt.title("Is the signal at the same relative depth across models?")
plt.tight_layout(); plt.savefig(ROOT / "model_comparison.png", dpi=160); plt.show()

## Stage 9 - Live demo app (Gradio)
Gives a public link you can open on any phone or laptop during the review.

In [ ]:
import gradio as gr, joblib

final_probe = fit(Xb, y)                              # train on all TriviaQA
joblib.dump({"probe": final_probe, "pooling": BEST, "layer": L, "model": MODEL_ID},
            WORK / "probe.joblib")

def ask(question, threshold):
    if not question.strip():
        return "", {}, ""
    p = build_prompt(question)
    enc = tok(p, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    ans = tok.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
    res = features_for(p, ans)
    if res is None:
        return ans, {}, "model gave no answer"
    feats, _, _ = res
    r = float(risk_of(final_probe, feats[BEST][L].reshape(1, -1).astype(np.float32))[0])
    verdict = (f"NOT CONFIDENT - would abstain (risk {r:.2f} > {threshold:.2f})" if r > threshold
               else f"confident - answer shown (risk {r:.2f} <= {threshold:.2f})")
    return ans, {"likely correct": 1 - r, "likely hallucinated": r}, verdict

gr.Interface(
    fn=ask,
    inputs=[gr.Textbox(label="Question"), gr.Slider(0, 1, value=0.6, step=0.05, label="Risk threshold")],
    outputs=[gr.Textbox(label="Model answer"), gr.Label(label="Probe says"), gr.Textbox(label="Decision")],
    title="Hallucination Probe",
    description=f"{MODEL_ID} - probe on layer {L} ({POOL_NAMES[BEST]}). One forward pass, no resampling.",
    examples=[["What is the capital of France?", 0.6],
              ["Who won the 1994 Nobel Prize in Literature?", 0.6],
              ["What is the population of Zaqatala district?", 0.6]],
).launch(share=True)

## Summary - the lines for the review

In [ ]:
r = results; b = r["best_pooling"]
print(f"Model {r['model']}: {r['n_layers']} layers, TriviaQA accuracy {100 * r['trivia_accuracy']:.1f}%")
print(f"Best probe: {POOL_NAMES[b]}, layer {r['best_layer']} ({100 * r['best_rel_depth']:.0f}% depth), "
      f"AUROC {r['robustness'][b]['mean']:.3f} +/- {r['robustness'][b]['std']:.3f} over {N_SEEDS} splits")
print(f"Before the answer exists (p_last): AUROC {r['robustness']['p_last']['mean']:.3f}")
bl = r["baselines"]
print(f"Log-prob baseline: {bl['min token log-prob']['mean']:.3f} | probe+log-prob: {bl['probe + log-prob']['mean']:.3f}")
print(f"Probe vs log-prob 95% CI: [{r['bootstrap_ci'][0]:+.3f}, {r['bootstrap_ci'][1]:+.3f}]")
print(f"Skip riskiest 20%: accuracy {100*(1-y[te].mean()):.1f}% -> {100*r['selective']['0.8']['probe']:.1f}% "
      f"({100*r['selective']['0.8']['captured']:.0f}% of a perfect detector)")
t = r["transfer_nq"]
print(f"Transfer TriviaQA -> NQ: {t['trivia_to_nq']:.3f} (in-domain NQ {t['nq_in_domain']:.3f})")
if "self_consistency" in r:
    s = r["self_consistency"]
    print(f"Self-consistency costs {s['sc_s_per_q'] / s['probe_s_per_q']:.0f}x the probe per question")